#Descrição do Notebook

Este Notebook apresenta um exemplo de aplicação de representações multimodais de médio nível semântico. As representações foram geradas para tomadas de vídeo, com intuito de utiliza-las na tarefa de segmentação automática de vídeo em cenas.

Assim, este Notebook contém a implementação de um segmentador de vídeo em cenas disponível na literatura. O segmentador utiliza as representações multimodais previamente computadas. Espera-se que essas representações, sendo sementicamente "mais ricas" que representações de baixo nível, ajudem a melhorar a eficácia na tarefa de segmentação.

#Preparação do ambiente do Notebook





**Instalando Dependências**

In [1]:
import networkx as nx
import numpy as np
from scipy.spatial import distance
import pickle
import pandas as pd

**Acesso ao Drive - para armazenamento de Arquivos**

In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


Mounted at /content/drive


<h1>Segmentador STG</h1>
(Scene Transition Graph - STG)

Implementação de um método clássico de segmentação de video em cenas baseado em grafos, onde cada nó do grafo representa uma tomada. Nesse modelo as transições são representadas como arestas de corte.

Por ser genérico e flexível quanto à implementação, o método STG tem sido reproduzido na literatura ao longo dos anos de diferentes modos e com diferentes propriedades adicionais.

Referência: YEUNG, M.; YEO, B.-L.; LIU, B. Segmentation of Video by Clustering and Graph Analysis.
Computer Vision and Image Understanding, v. 71, n. 1, p. 94–109, jul 1998. ISSN 10773142.
Disponível em: <https://linkinghub.elsevier.com/retrieve/pii/S1077314297906287>.

In [3]:
class STG:

    def __init__(self, multiModalHistogram, delta=0.35, timeThreshold=7):
        self.delta = delta
        self.timeThreshold = timeThreshold

        nShots = len(multiModalHistogram)

        self.interShotDistances = np.zeros((nShots, nShots))
        for i in range(nShots):
            for j in range(i+1, nShots):
                self.interShotDistances[i][j] = self.interShotDistances[j][i] = distance.cosine(multiModalHistogram[i], multiModalHistogram[j])

    def clusterDistance(self, clusterA, clusterB):
        maxDist = 0
        for shotI in clusterA:
            for shotJ in clusterB:
                if self.interShotDistances[shotI][shotJ] > maxDist:
                    if abs(shotI - shotJ) > self.timeThreshold:
                        maxDist = float('inf')
                    else:
                        maxDist = self.interShotDistances[shotI][shotJ]

        return maxDist

    def allClustersDistancesGreaterThanDelta(self):
        nClusters = len(self.clusters)
        for i in range(nClusters):
            for j in range(i+1, nClusters):
                if self.clusterDistance(self.clusters[i], self.clusters[j]) <= delta:
                    return False
        return True

    def linkageCluster(self):
        nClusters = len(self.interShotDistances)
        self.clusters = [[i] for i in range(nClusters)]

        while not self.allClustersDistancesGreaterThanDelta() and nClusters > 1:
            minDist = self.interShotDistances[0][1]
            clusterR = 0
            clusterS = 1
            for i in range(nClusters):
                for j in range(i+1, nClusters):
                    currentDist = self.clusterDistance(self.clusters[i], self.clusters[j])
                    if currentDist < minDist:
                        minDist = currentDist
                        clusterR = i
                        clusterS = j

            # Merge R and S into a new cluster
            for shot in self.clusters[clusterS]:
                self.clusters[clusterR].append(shot)
            self.clusters.pop(clusterS)
            nClusters -= 1

    def containsSubsequentShots(self, clusterA, clusterB):
        for shotI in clusterA:
            for shotJ in clusterB:
                if shotI == shotJ + 1 or shotJ == shotI + 1:
                    return True

        return False

    def buildGraph(self):
        # STG Construction
        nClusters = len(self.clusters)
        self.g = nx.Graph()
        for i in range(nClusters):
            self.g.add_node(i)

        for i in range(nClusters):
            for j in range(i+1, nClusters):
                if self.containsSubsequentShots(self.clusters[i], self.clusters[j]):
                    self.g.add_edge(i, j)

        bridges = nx.bridges(self.g)
        for bridge in bridges:
            self.g.remove_edge(bridge[0], bridge[1])

    def segment(self):
        connectedComponents = list(nx.connected_components(self.g))
        nConnectedComponents = len(connectedComponents)
        for i in range(nConnectedComponents):
            for node in connectedComponents[i]:
                self.g.nodes[node]['scene'] = i

        # Divide shots into scenes
        scenes = [[] for i in range(nConnectedComponents)]
        for node in self.g.nodes.data():
            scenes[node[1]['scene']].append(node[0])

        self.scenesBoundaries = np.empty((nConnectedComponents, 2), dtype=np.int32)
        for i in range(nConnectedComponents):
            shotsInScene = []
            for clusterIndex in scenes[i]:
                shotsInScene += self.clusters[clusterIndex]
            shotsInScene = sorted(shotsInScene)
            self.scenesBoundaries[i][0] = shotsInScene[0] + 1
            self.scenesBoundaries[i][1] = shotsInScene.pop() + 1



#Cálculo das métricas de eficácia

Neste exemplo são usadas as métricas *Coverage*, *New Overflow* e F1. São métricas específicas para medir eficácia de algoritmos de segmentação temporal de vídeo (em tomdas e especialmente em cenas) [1]. Essas métricas foram otimizadas a partir de medidas anteriores (*Coverage* e *Overflow*) [2]. Deseja-se um valor alto para *Coverage* e um valor baixo para *New Overflow*.

Referências:

[1] Vendrig J, Worring M (2002) Systematic evaluation of logical story unit segmentation. IEEE Trans
Multimedia 4(4):492–499. https://doi.org/10.1109/TMM.2002.802021

[2] Han B, Wu W (2011) Video scene segmentation using a novel boundary evaluation criterion and
dynamic programming. In: 2011 IEEE International conference on multimedia and expo, pp 1–6.
https://doi.org/10.1109/ICME.2011.6012001








In [4]:
def getNewOverflow(gt, seg):
    newOverflowForScene = []
    for boundaryGt in gt:
        soma = 0
        for boundarySeg in seg:
            if boundarySeg[0] > boundaryGt[-1]:
                break
            intersection = len(set.intersection(set(boundaryGt), set(boundarySeg)))
            if intersection > 0:
                soma += len(boundarySeg)
        no = 0
        if soma != 0:
            subtraendo = len(boundaryGt) / soma
            # Only for debug purposes
            #if subtraendo > 1:
            #    print('NEGATIVE NEW OVERFLOW')
            no = 1 - subtraendo
        newOverflowForScene.append(no)

    nShots = gt[-1][-1]

    avgNewOverflow = 0
    for i in range(len(gt)):
        avgNewOverflow += (newOverflowForScene[i] * len(gt[i]) / nShots)

    return (1-avgNewOverflow)


def getCoverage(gt, seg):
    '''
    gt and seg are sets containing the first and the last shot of every scene in, respectively,
    the ground truth annotation and segmentation algorithm output
    '''
    coverageForScene = []
    for boundaryGt in gt:
        maior = 0
        for boundarySeg in seg:
            if boundarySeg[0] > boundaryGt[-1]:
                break
            maior = max(maior, len(set.intersection(set(boundaryGt), set(boundarySeg))))
        coverageForScene.append(maior / len(boundaryGt))

    nShots = gt[-1][-1]

    avgCoverage = 0
    for i in range(len(gt)):
        avgCoverage += (coverageForScene[i] * len(gt[i])/nShots)

    return avgCoverage


def toSet(input):
    output = []
    for i in input:
        output.append(range(i[0], i[1]+1))
    return output

def getMetrics(scenesBoundaries, groundTruthFile):
    predictedBoundaries = toSet(scenesBoundaries)

    # ler groundTruth de arquivo
    scenes = pd.read_csv(groundTruthFile)
    groundTruth = list((boundary[0], boundary[1]) for _,boundary in scenes.iterrows())
    groundTruth = toSet(groundTruth)

    Coverage = getCoverage(groundTruth, predictedBoundaries)
    NewOverflow = getNewOverflow(groundTruth, predictedBoundaries)
    F1 = 2 * (Coverage * NewOverflow) / (Coverage + NewOverflow)

    return Coverage, NewOverflow, F1

def getMetrics_2(scenesBoundaries, groundTruthFile, shotTolerance=2):
    predictedBoundaries = toSet(scenesBoundaries)

    # ler groundTruth de arquivo
    scenes = pd.read_csv(groundTruthFile)
    groundTruth = list((boundary[0], boundary[1]) for _,boundary in scenes.iterrows())
    groundTruth = toSet(groundTruth)

    p,r,fpr = getPrecisionAndRecall(groundTruth, predictedBoundaries, shotTolerance)

    return p,r,fpr

def getPrecisionAndRecall(gt, seg, shotTolerance):

    gt = set(boundary[-1] for boundary in gt)
    seg = set(boundary[-1] for boundary in seg)
    gtWithTolerance = gt.copy()
    for boundary in gt:
        for i in range(boundary - shotTolerance, boundary + shotTolerance + 1):
            if i>=1: gtWithTolerance.add(i)

    vp = len(gtWithTolerance.intersection(seg))
    fp = len(seg.difference(gt))
    fn = len(gt.difference(seg))

    precision = vp / (vp + fp)
    recall = vp / (vp + fn)
    f1 = 2 * precision * recall / (precision + recall)

    return precision, recall, f1



#Script para Segmentação

In [57]:
##Script Principal (aka _main_)

delta = 0.35
timeThreshold = 7

video = 'bbc_11'
tamDic = '100'
#operador = 'sum'

#inputFile = 'mm_sum_histogram_bbc_01_100.arq'
#inputFile = 'mm_sum_histogram_bbc_01_100_cosine_leticia.arq'
#inputFile = 'aural_histogram_100_bbc_01.mp4.arq'
#+operador+'_histogram_'+video+'_'+tamDic+'.arq'
outFile = video+'_'+tamDic+'_segmentation.arq'

with open('/content/drive/MyDrive/DatasetBBC_Leticia/Histogramas/Histogramas_Visuais/visual_histogram_100_cosine_bbc_11.arq' ,'rb') as arq:
    mmHistogram = pickle.load(arq)

print(mmHistogram[0], mmHistogram[1])

stg = STG(mmHistogram, delta, timeThreshold)
stg.linkageCluster()
stg.buildGraph()
stg.segment()


with open('/content/drive/MyDrive/DatasetBBC_Leticia/Segmentation/visual/'+outFile,'wb') as arq:
    pickle.dump(stg.scenesBoundaries, arq)

print("Segmentação concluída")
#print(stg.scenesBoundaries)

#Cálculo de eficácia
#c,no,f1 = getMetrics(stg.scenesBoundaries,'/content/drive/MyDrive/DatasetBBC_Leticia/GT/bbc_01_scenes.csv')
#print('Coverage: ', c, 'New Overflow: ', no, 'F1: ', f1)

p,r,fpr = getMetrics_2(stg.scenesBoundaries,'/content/drive/MyDrive/DatasetBBC_Leticia/GT/bbc_11_scenes.csv')
print('P: ', p, 'R: ', r, 'F1: ', fpr)


[0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 2. 0. 0. 0. 0. 0. 3. 0. 1. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 1. 0. 0. 4. 1. 2. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 4. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 2. 0. 0. 0. 0.
 0. 0. 2. 0. 0. 0. 0. 0. 0. 0. 2. 1. 0. 0. 0. 0. 1. 0. 0. 6. 0. 0. 0. 1.
 0. 0. 0. 0.] [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 6. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 6. 0.]
Segmentação concluída
P:  0.45714285714285713 R:  0.24242424242424243 F1:  0.31683168316831684


<ipython-input-4-94a01d226a93>:77: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  groundTruth = list((boundary[0], boundary[1]) for _,boundary in scenes.iterrows())


In [ ]:
from google.colab import drive
drive.mount('/content/drive')